## xLH-ai-bp-vs-hf
Eine exemplarische Umsetzung für die Integration von LLM-Modellen für die Berufspädagogik

In [1]:
import os
import pathlib
import sys
__cwd__ = str(pathlib.Path(os.getcwd())).replace('\\\\', '/')
sys.path.append(__cwd__)
print(__cwd__)


D:\Python\xLH-mims\notebooks\src\ai_bp_hf


In [4]:
import pandas as pd
from dataclasses import dataclass
from pydantic_ai import Tool
from rich import print
from pydantic import BaseModel, Field
from pydantic_ai import Agent
from tools import set_oven_temperature
from llm_model import get_model, LlmModel
from object_to_file import base_model_to_file
from config import config
import nest_asyncio
nest_asyncio.apply()

In [3]:
# OpenAi API Key (Credentials hinterlegt in der Datei ..\xLH-mims-data\config\xlh_mims_python.env)
print(config.openai_api_key[:20])
# falls keine Datei im lokalen Speicher vorhanden ist, wir die Datei env.py

sk-proj-hevDtsISJBCJ

In [7]:
# df = pd.read_pickle("zusammenstellung_RS_LK.pkl")
# df.head()

,id_hk,Niveau IPRE,id_lk,Leistungskriterien
0,rs_hk_01_01,I,ts_lk_01_01_001,Dipl. RS HF handeln in dieser Situation kompet...
1,rs_hk_01_01,I,ts_lk_01_01_002,Dipl. RS HF handeln in dieser Situation kompet...
2,rs_hk_01_01,I,ts_lk_01_01_003,Dipl. RS HF handeln in dieser Situation kompet...
3,rs_hk_01_01,P,ts_lk_01_01_004,Dipl. RS HF handeln in dieser Situation kompet...
4,rs_hk_01_01,P,ts_lk_01_01_005,Dipl. RS HF handeln in dieser Situation kompet...


In [5]:
@dataclass
class Deps:
    name: str

In [6]:
class Quantity(BaseModel):
    number_of_operational_competence: str = Field(description='Anzahl der Handlungskompetenzen')
    number_of_performance_criterion: str = Field(description='Anzahl der Leistungskriterien')

In [8]:
model = get_model(llm_model=LlmModel.OPENAI_GPT_5_2)  # ToDo: Integration OLLAMA...
agent= Agent(
    model=model,
    system_prompt=('Du bist ein Pizzabäcker welcher Rezepte für kreative Pizzas kreirt. '
                   'Nutze das Tool set_oven_temperature für die Einstellung des Backofens.'),
    deps_type=Deps,
    tools=[
        Tool(set_oven_temperature, takes_ctx=True),
    ],
    retries=3,
    output_type=Recipe,
)
deps = Deps(name='-')

In [9]:
response = agent.run_sync(user_prompt='Ich habe im Kühlschrank Lachs, Salami und Tomaten '
                                 'Kreire mir eine Pizza. '
                                 'Da mein Backofen etwas alterschwach ist, '
                                 'kann die Temperatur nicht höher als 225 Grad eingestellt werden '
                                 'und die Backzeit darf 30 Minuten nicht überschreiten.')

=> tool_set_oven_temperature 225.0
=> set_oven_temperature_opcua 225.0


In [10]:
result: Recipe = response.output
base_model_to_file(result)
# siehe recipe.json
print(f'Antwort: {result.model_dump_json(indent=4)[:100]}')

Antwort: {
    "title": "Pizza Lachs-Salami mit frischen Tomaten",
    "ingredients": [
        {

In [11]:
usage = response.usage()
print(f'Input Tokens: {usage.input_tokens} ({usage.input_tokens*1.75*1e-6:0.04f} $)')
print(f'Output Tokens: {usage.output_tokens} ({usage.output_tokens*14.0*1e-6:0.04f} $)')
print(f'Total Tokens: {usage.total_tokens} (total cost: {(usage.input_tokens*1.75*1e-6 + usage.input_tokens*1.75*1e-6):0.04f} $)')

Input Tokens: 524 (0.0009 $)

Output Tokens: 403 (0.0056 $)

Total Tokens: 927 (total cost: 0.0018 $)

In [12]:
settings = agent.model.settings
print(f'Settings: {settings}')
print(f'temperature: {settings.get('temperature')}')
print(f'presence_penalty: {settings.get('presence_penalty')}')
print(f'frequency_penalty: {settings.get('frequency_penalty')}')

Settings: {'temperature': 0.0, 'presence_penalty': 0.0, 'frequency_penalty': 0.0, 'parallel_tool_calls': True, 
'openai_reasoning_effort': 'none', 'openai_reasoning_summary': 'auto', 'openai_text_verbosity': 'low'}

temperature: 0.0

presence_penalty: 0.0

frequency_penalty: 0.0